## Imports

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)
local_datasets = '/Users/rapido/local-datasets/customer/appography/appography_v1/'
print(local_datasets)

/Users/rapido/analytics/customer/Appography/Appography_v1
/Users/rapido/local-datasets/customer/appography/appography_v1/


### customer installed apps

#### Old Logic 

Scaling started from 20240219

##customer_installed_apps

customer_installed_apps_query = f"""

    SELECT
        *
    FROM
        (
        SELECT
            id user, 
            yyyymmdd,
            json_extract(data, '$.appographyData') as app_list,
            row_number() over(partition by id order by updated_epoch desc) updated_seq
        FROM 
            raw_internal.kafka_info_customer_installed_apps_v1
        WHERE 
            yyyymmdd >= DATE_FORMAT((DATE_TRUNC('month', CURRENT_DATE) - INTERVAL '2' MONTH), '%Y%m%d')
            AND yyyymmdd <= DATE_FORMAT((CURRENT_DATE - INTERVAL '1' DAY), '%Y%m%d')
        )
    WHERE
        updated_seq = 1
"""

df_customer_installed_apps = pd.read_sql(customer_installed_apps_query, connection)
df_customer_installed_apps.to_csv(local_datasets + 'raw/raw_customer_installed_apps.csv', index=False)

In [ ]:
start_time = time.time()

df_customer_installed_apps = pd.read_csv(local_datasets + 'raw/raw_customer_installed_apps.csv')
df_customer_installed_apps_v1 = df_customer_installed_apps[['user', 'app_list']].copy()

##Define a function to extract values between double quotations and trim extra spaces
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r'"(.*?)"', text)]

df_customer_installed_apps_v1['app_list'] = df_customer_installed_apps_v1['app_list'].apply(lambda x: extract_and_trim(x))

end_time = time.time()
execution_time = ((end_time - start_time)/60.00)
print("Execution time:", execution_time, "mins")
print(df_customer_installed_apps_v1.shape)
df_customer_installed_apps_v1.head(1)

#### New Logic

In [5]:
combined_df_latest = pd.read_csv('/Users/rapido/local-datasets/customer/appography/appography_data_last_sync/appography-data-march.csv')

In [6]:
combined_df_latest = combined_df_latest[['user_id', 'app_list_set']]
combined_df_latest.head(5)

,user_id,app_list_set
0,5737c6acddbec2203f73315e,"['flipkart', 'disney+ hotstar', 'messenger', '..."
1,5737c6adddbec2203f73316d,"['whatsapp', 'paytm money', 'samsung galaxy we..."
2,5737c6aeddbec2203f733176,"['outlook', 'onedrive', 'ola', 'microsoft 365'..."
3,5737c6b0ddbec2203f733182,"['wynk music', 'messenger', 'mx takatak', 'goo..."
4,5737c6b1ddbec2203f73318b,"['snapchat', 'instagram', 'google photos', 'zo..."


### category list

Google Sheet - https://docs.google.com/spreadsheets/d/1O6IsJ2dzzhXKuoHFoOA0AD4s4SOerMC1U-KrpeFnbwQ/edit#gid=1742529009

category_list = pd.read_clipboard()
category_list.to_csv(local_datasets + 'category_list_v1.csv', index=False)

In [7]:
category_list = pd.read_csv(local_datasets + 'category_list_v1.csv')
category_list['app_name'] = category_list['app_name'].str.lower()
category_list = category_list[['app_name', 'categories_l2', 'categories_l1']]
print(category_list.shape)
print(category_list.categories_l1.unique())
print(category_list.categories_l2.unique())
category_list.head(1)

(399, 3)
['Educational' 'OTT' 'Delivery_Grocery' 'Tools' 'Office' 'Dating' 'News'
 'Devotional' 'Travel_Metro' 'Gaming' 'Travel_Bookings'
 'Finance_Transactions' 'Ecommerce' 'Streaming_Music' 'Finance_Investment'
 'Health' 'Fantasy_Sports' 'Delivery_Food' 'Home_Security' 'Finance_News'
 'Travel_Ridehailing' 'Social' 'Telecom' 'Entertainment' 'News_finance'
 'Wearable' 'Driver_App']
['Educational' 'Rest' 'Office' 'Ride haling' 'Gaming' 'Finance_Investment'
 'Driver_App']


,app_name,categories_l2,categories_l1
0,ignou e-content,Educational,Educational


### Active customer (RR-customers)

In [8]:
## Parameter 
start_date = '20240318'
end_date = '20240331'

##active_customer_query

active_customer_demographic_query = f"""

    WITH active_customers AS (

    SELECT 
        channel_host,
        customer_id,
        COALESCE(city_name, 'NA') city_name
    FROM 
        (
        SELECT 
            channel_host,
            customer_id,
            city_name,
            row_number() over(partition by customer_id order by updated_epoch DESC) updated_seq
        FROM 
            orders.order_logs_snapshot
        WHERE
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND channel_host IN ('android', 'ios')
            AND city_name IN ('Bangalore', 'Hyderabad', 'Chennai', 'Delhi', 'Jaipur', 'Indore')
        ) 
    WHERE
        updated_seq = 1
    ), 

    customer_single_view AS (

    SELECT
        customerId,
        gender,
        lifetimeage rapido_age,
        CASE 
        WHEN ((POSITION('/' IN dateofbirth) > 0 AND dateofbirth <> 'NaN/NaN/NaN') or POSITION('-' IN dateofbirth) > 0) 
        THEN 
            CASE 
            WHEN TRY(date_diff('year', CAST(dateofbirth AS DATE), current_date)) IS NOT NULL 
            THEN date_diff('year', CAST(dateofbirth AS DATE), current_date)
            ELSE NULL
            END
        ELSE NULL
        END AS customer_age,
        lifetimerides
    FROM 
        datasets.customer_single_view
    WHERE 
        customerId IN (SELECT customer_id FROM active_customers)
    ),

    iallocator_customer_segments AS (

    SELECT
        customer_id user_id,
        taxi_ltr_segment ltr_segment,
        taxi_lifetime_rides life_time_rides,
        taxi_lifetime_stage life_time_stage,
        taxi_income_segment income_segment,
        taxi_service_affinity service_affinity,
        taxi_distance_preference distance_preference,
        CASE 
        WHEN ps_tag_auto = ps_tag_link THEN ps_tag_auto
        WHEN ps_tag_auto = 'PS' OR ps_tag_link = 'PS' THEN 'PS'
        WHEN ps_tag_auto = 'NPS' OR ps_tag_link = 'NPS' THEN 'NPS'
        ELSE 'UNKNOWN'
        END ps_tag_taxi

    FROM 
        datasets.iallocator_customer_segments
    WHERE 
        run_date = DATE_FORMAT(DATE_PARSE('{end_date}', '%Y%m%d'), '%Y-%m-%d')
        AND customer_id IN (SELECT customer_id FROM active_customers)
    )

    SELECT
        a.*,
        c.gender,
        c.rapido_age,
        COALESCE(CAST(c.customer_age AS VARCHAR), 'NA') customer_age,
        CASE 
        WHEN life_time_rides = 0 OR c.lifetimerides = 0 THEN 'LTR 0'
        WHEN life_time_rides > 0 THEN ltr_segment
        ELSE 'UNKNOWN'
        END AS ltr_segment,
        COALESCE(life_time_rides, c.lifetimerides) life_time_rides,
        COALESCE(life_time_stage, 'UNKNOWN') AS life_time_stage,
        COALESCE(income_segment, 'UNKNOWN') AS income_segment,
        COALESCE(service_affinity, 'NO_AFFINITY') AS service_affinity,
        COALESCE(distance_preference, 'UNKNOWN') AS distance_preference,
        COALESCE(ps_tag_taxi, 'UNKNOWN') AS price_sensitivity 
    FROM 
        active_customers a
    LEFT JOIN 
        customer_single_view c
        ON a.customer_id = c.customerId
    LEFT JOIN 
        iallocator_customer_segments b
        ON a.customer_id = b.user_id
"""

df_active_customer_demographic = pd.read_sql(active_customer_demographic_query, connection)
df_active_customer_demographic.to_csv(local_datasets + 'raw/active_customer_demographic_v1.csv', index=False)


active_customer_ao_net_query = f"""

    WITH fe_tbl AS (
        SELECT
            DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
            city,
            service_name,
            user_id as user_id,
            fare_estimate_id
        FROM
            hive.pricing.fare_estimates_enriched
        WHERE 
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND city IN ('Bangalore', 'Hyderabad', 'Chennai', 'Delhi', 'Jaipur', 'Indore')
    ),

    rr_net_tbl as (
        SELECT   
            DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
            city_name AS city,
            service_obj_service_name AS service_name,
            customer_id,
            estimate_id AS fare_estimate_id,
            order_id,
            order_status,
            spd_fraud_flag
        FROM orders.order_logs_snapshot ols
        WHERE
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND channel_host IN ('android', 'ios')
            AND city_name IN ('Bangalore', 'Hyderabad', 'Chennai', 'Delhi', 'Jaipur', 'Indore')
    ),

    fe_merged AS (
        SELECT
            fe_tbl.*,
            rr_net_tbl.customer_id,
            rr_net_tbl.order_id,
            rr_net_tbl.order_status,
            rr_net_tbl.spd_fraud_flag
        FROM
            fe_tbl
        LEFT JOIN
            rr_net_tbl
            ON fe_tbl.order_date = rr_net_tbl.order_date
            AND fe_tbl.user_id = rr_net_tbl.customer_id
    )

    SELECT
        customer_id,
        COUNT(distinct fare_estimate_id) AS fe_count,
        COUNT(distinct order_id) AS rr_count,
        COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END) AS net_count

    FROM 
        fe_merged
    GROUP BY 1

"""

df_active_customer_ao_net_query = pd.read_sql(active_customer_ao_net_query, connection)
df_active_customer_ao_net_query.to_csv(local_datasets + 'raw/active_customer_ao_net_v1.csv', index=False)

In [9]:
df_active_customer_demographic = pd.read_csv(local_datasets + 'raw/active_customer_demographic_v1.csv')
df_active_customer_ao_net_query = pd.read_csv(local_datasets + 'raw/active_customer_ao_net_v1.csv')

##active_customer_demographic & funnel
df_active_customer = pd.merge(df_active_customer_demographic,
                              df_active_customer_ao_net_query,
                              how='left',
                              on = ['customer_id']
                             )

#df_active_customer['fe2rr']=(df_active_customer['rr_count']*100.00/df_active_customer['fe_count']).round(2)
#df_active_customer['g2n']=(df_active_customer['net_count']*100.00/df_active_customer['rr_count']).round(2)
#df_active_customer['fe2net']=(df_active_customer['net_count']*100.00/df_active_customer['fe_count']).round(2)

df_active_customer.to_csv(local_datasets + 'processed/active_customer_v1.csv', index=False)

##reducing load
df_active_customer = df_active_customer.iloc[0:0]
df_active_customer_demographic = df_active_customer_demographic.iloc[0:0]
df_active_customer_ao_net_query = df_active_customer_ao_net_query.iloc[0:0]

In [10]:
df_active_customer = pd.read_csv(local_datasets + 'processed/active_customer_v1.csv')

## Data workaround

In [11]:
print(df_active_customer.shape)
df_active_customer['gender'] = df_active_customer['gender'].fillna('UNKNOWN')
df_active_customer.head(1)

(6127583, 16)


,channel_host,customer_id,city_name,gender,rapido_age,customer_age,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,fe_count,rr_count,net_count
0,android,6370bc7bee51da794a7f7727,Hyderabad,MALE,511.0,NaN,PHH,75,COMMITTED,MEDIUM_INCOME,ONLY_LINK,SHORT_DISTANCE,NPS,12.0,8.0,8.0


In [12]:
df_customer_installed_apps_v1 = combined_df_latest[combined_df_latest['user_id'].isin(df_active_customer['customer_id'])]

In [13]:
print(df_customer_installed_apps_v1.shape)
df_customer_installed_apps_v1.head(1)

(4656065, 2)


,user_id,app_list_set
2,5737c6aeddbec2203f733176,"['outlook', 'onedrive', 'ola', 'microsoft 365'..."


In [14]:
print(category_list.shape)
category_list.head(1)

(399, 3)


,app_name,categories_l2,categories_l1
0,ignou e-content,Educational,Educational


### What is the distribution of mobile os among our customer base?

In [15]:
df_temp = df_active_customer.groupby(['channel_host']).agg({'customer_id':'nunique'}).reset_index()
df_temp['% distribution'] = (df_temp['customer_id']*100/df_temp['customer_id'].sum()).round(2)
df_temp### Which city should I choose?

,channel_host,customer_id,% distribution
0,android,5098301,83.2
1,ios,1029282,16.8


### Which city should I choose?

In [16]:
df_active_customer_android = df_active_customer[df_active_customer['channel_host'].isin(['android'])]

df_customer_coverage = pd.merge(df_active_customer_android,
                                df_customer_installed_apps_v1,
                                how='left',
                                left_on=['customer_id'], 
                                right_on=['user_id']
                                )

In [18]:
df_temp = df_customer_coverage\
            .groupby(['city_name'])\
            .agg(active_customers = pd.NamedAgg('customer_id','nunique'),
                 app_info_customers = pd.NamedAgg('user_id','nunique')
                )\
            .sort_values(['active_customers'],ascending=False)\
            .reset_index()
df_temp['coverage %'] = (df_temp['app_info_customers']*100/df_temp['active_customers']).round(2)
df_temp.head(20)

,city_name,active_customers,app_info_customers,coverage %
0,Hyderabad,1758960,1614552,91.79
1,Bangalore,1132598,1024954,90.50
2,Delhi,1038175,957065,92.19
3,Chennai,774176,690685,89.22
4,Jaipur,247066,220564,89.27
5,Indore,147326,130595,88.64


In [19]:
# df_temp.to_csv('city_app_customer_coverage.csv', index=False)

### Work-around

### Proceeding with Bangalore city

In [122]:
## Merge Active customers & customers apps details

df_customer_ban_all = df_customer_coverage[(df_customer_coverage['city_name'].isin(['Indore'])) 
                                           &
                                           (df_customer_coverage['user_id'].notna())]


df_customer_ban_all = df_customer_ban_all[['customer_id', 'gender', 'rapido_age', 'customer_age',
                                           'fe_count', 'rr_count', 'net_count', 
                                           'ltr_segment', 'life_time_rides', 'life_time_stage',
                                           'income_segment', 'service_affinity', 'distance_preference',
                                           'price_sensitivity', 'app_list_set']]

##  Derived attribute RPC
df_customer_ban_all['net_count_with_nan'] = df_customer_ban_all['net_count'].replace(0, np.nan)

df_customer_ban_all['rpc'] = np.where(df_customer_ban_all['net_count'] < 1, 'a. zero rpc',
                                np.where(df_customer_ban_all['net_count'] < 2, 'b. low rpc',
                                    np.where(df_customer_ban_all['net_count'] < 4, 'c. medium rpc', 
                                             'd. high rpc')))

df_customer_ban_all.to_csv(local_datasets + 'city/indore/raw/indore_customers_v1.csv', index=False)
print(df_customer_ban_all.shape)
df_customer_ban_all.head(1)

(130595, 17)


,customer_id,gender,rapido_age,customer_age,fe_count,rr_count,net_count,ltr_segment,life_time_rides,life_time_stage,income_segment,service_affinity,distance_preference,price_sensitivity,app_list_set,net_count_with_nan,rpc
119,5e3b93c0d6d3936cd767fbdb,MALE,255.0,30.0,6.0,3.0,2.0,PHH,14,HOOK,HIGH_INCOME,ONLY_AUTO,LONG_DISTANCE,NPS,"['ms edge', 'whatsapp', 'paytm money', 'google...",2.0,c. medium rpc


In [123]:
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_customer_ban_all['app_list_set'] = df_customer_ban_all['app_list_set'].apply(lambda x: extract_and_trim(x))

In [124]:
total_customers = df_customer_ban_all.customer_id.nunique()
print(total_customers)

130595


### Customer level apps list and category mapping 

In [125]:
df_customer_app_list = df_customer_ban_all[['customer_id', 'app_list_set']]
df_customer_app_list.reset_index(drop=True, inplace=True)

df_customer_app_list_explode = df_customer_app_list.explode('app_list_set')
df_customer_app_list_explode['app_list_set'] = df_customer_app_list_explode['app_list_set'].str.lower()

df_customer_app_list_explode.head(1)

,customer_id,app_list_set
0,5e3b93c0d6d3936cd767fbdb,ms edge


#### Customer-wise App list

In [126]:
total_customers = df_customer_app_list_explode.customer_id.nunique()
total_customers

130595

In [127]:
df_temp = df_customer_app_list_explode\
            .groupby(['app_list_set'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
print(df_customer_app_list_explode.app_list_set.nunique())
df_temp

213


,app_list_set,customers,%
0,google maps,130537,99.96
1,youtube,130358,99.82
2,gmail,130343,99.81
3,whatsapp,125338,95.97
4,google photos,122153,93.54
5,instagram,109338,83.72
6,truecaller,94686,72.50
7,paytm money,84366,64.60
8,snapchat,81747,62.60
9,google calendar,81115,62.11


In [128]:
# df_temp.to_csv('bangalore_app_wise_customer_distribution.csv', index=False)

#### Consider removing commonly used apps

In [129]:
commonly_used_app_list = ['youtube','google maps','gmail','whatsapp','google photos','google calendar']
commonly_used_app_list

['youtube',
 'google maps',
 'gmail',
 'whatsapp',
 'google photos',
 'google calendar']

In [130]:
df_customer_app_cat_mapping = pd.merge(df_customer_app_list_explode[~df_customer_app_list_explode['app_list_set']\
                                                                    .isin(commonly_used_app_list)], 
                                          category_list, 
                                          how='left', left_on='app_list_set', right_on='app_name')
df_customer_app_cat_mapping = df_customer_app_cat_mapping.drop_duplicates()
df_customer_app_cat_mapping.to_csv(local_datasets + 'city/indore/processed/customer_app_categories_mapping_v1.csv',
                                   index=False)

In [131]:
df_customer_app_cat_mapping = pd.read_csv(local_datasets 
                                          + 'city/indore/processed/customer_app_categories_mapping_v1.csv')

df_customer_mapped = df_customer_app_cat_mapping\
                        .groupby(['customer_id'])\
                        .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                             app_count = ('app_list_set', 'nunique'),
                             categories_l1 = ('categories_l1', lambda x: list(set(x))),
                             categories_l1_count = ('categories_l1', 'nunique'),
                             categories_l2 = ('categories_l2', lambda x: list(set(x))),
                             categories_l2_count = ('categories_l2', 'nunique')
                            )\
                        .reset_index()

# print('total_customers -', total_customers)
print(df_customer_mapped.shape)
df_customer_mapped.head(5)

(130491, 7)


,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
0,573f28f59b0ffc283676f94b,"[zomato, bookmyshow, kite by zerodha, uber, ju...",20,"[Tools, Educational, Delivery_Food, OTT, Enter...",10,"[Educational, Ride haling, Finance_Investment,...",4
1,573f28fe9b0ffc28367719f1,"[zoom, linkedin, microsoft teams, netflix, axi...",25,"[Tools, Social, News, OTT, Delivery_Grocery, F...",10,"[Ride haling, Office, Rest]",3
2,573f29089b0ffc2836773ef0,"[telegram, zoom, cred, zomato, messenger, indi...",30,"[Tools, Social, News, Educational, Delivery_Fo...",12,"[Educational, Finance_Investment, Office, Ride...",5
3,573f290e9b0ffc2836775ef5,"[telegram, zomato, linkedin, indigo, netflix, ...",23,"[Tools, Fantasy_Sports, Social, Delivery_Food,...",12,"[Ride haling, Office, Rest]",3
4,573f292a9b0ffc283677b413,"[instagram, facebook, kite by zerodha, zomato,...",6,"[OTT, Social, Finance_Investment, Delivery_Food]",4,"[Finance_Investment, Rest]",2


In [132]:
total_customers = df_customer_mapped.customer_id.nunique()
total_customers

130491

#### Distribution check on categories

In [133]:
df_temp = df_customer_app_cat_mapping\
            .groupby(['categories_l2'])\
            .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                 app_count = ('app_list_set', 'nunique'),
                 customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list,app_count,customers,%
0,Rest,"[amazon music, fitbit, tripadvisor, linkedin, ...",121,130450,99.97
1,Office,"[miro, zoom, dropbox, github, trello, pocket, ...",22,60031,46.00
2,Ride haling,"[driveu driver app, blusmart, quick ride, namm...",9,53987,41.37
3,Finance_Investment,"[angel broking, kite by zerodha, hdfc mobile b...",6,45953,35.22
4,Educational,"[course hero, mycbseguide, google classroom, n...",39,29021,22.24
5,Gaming,[ludo king],1,25116,19.25
6,Driver_App,"[rapido captain, shadowfax driver app, zepto d...",10,9071,6.95


In [134]:
df_temp = df_customer_app_cat_mapping\
            .groupby(['categories_l1'])\
            .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                 app_count = ('app_list_set', 'nunique'),
                 customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,app_count,customers,%
0,Social,"[skype, telegram, instagram, facebook, flipboa...",17,127811,97.95
1,Tools,"[microsoft lens, ms edge, google sheets, file ...",18,115431,88.46
2,OTT,"[zee5, alt balaji, aha, amazon prime video, so...",9,98643,75.59
3,Ecommerce,"[nykaa, myntra, pharmeasy, club factory, zivam...",13,94737,72.60
4,Finance_Transactions,"[axis mobile, cred, freecharge, mobikwik, payt...",5,88910,68.13
5,Streaming_Music,"[soundcloud, gaana, wynk music, amazon music, ...",6,80297,61.53
6,Delivery_Food,"[swiggy, uber eats, dunzo, zomato]",4,73068,55.99
7,Office,"[miro, zoom, dropbox, github, trello, pocket, ...",22,60031,46.00
8,Travel_Ridehailing,"[driveu driver app, blusmart, quick ride, namm...",9,53987,41.37
9,Finance_Investment,"[angel broking, kite by zerodha, hdfc mobile b...",6,45953,35.22


#### Category-level App usage 

In [135]:
print(df_customer_app_cat_mapping['categories_l1'].unique())

['Tools' 'Finance_Transactions' 'Delivery_Food' 'OTT' 'Social' 'Office'
 'Finance_Investment' 'News' 'Travel_Ridehailing' 'Educational' 'Gaming'
 'Streaming_Music' 'Entertainment' 'Travel_Bookings' 'Ecommerce'
 'Delivery_Grocery' 'Wearable' 'Dating' 'Finance_News' 'Fantasy_Sports'
 'Driver_App' 'Health' 'Devotional']


In [136]:
# single_category = ['Travel_Ridehailing']

# ### Office
# print(single_category)
# df_temp = df_customer_app_cat_mapping[df_customer_app_cat_mapping['categories_l1'].isin(single_category)]\
#             .groupby(['categories_l1','app_list'])\
#             .agg(customers = ('customer_id','nunique'))\
#             .sort_values(['customers'],ascending=False)\
#             .reset_index()
# df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
# df_temp

In [137]:
### Office
df_temp = df_customer_app_cat_mapping[~df_customer_app_cat_mapping['categories_l2'].isin(['Rest'])]\
            .groupby(['categories_l2','app_list_set'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(by=['categories_l2', 'customers'], ascending=[False, False])\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list_set,customers,%
0,Ride haling,ola,40179,30.79
1,Ride haling,uber,23002,17.63
2,Ride haling,jugnoo,11885,9.11
3,Ride haling,in drive,2035,1.56
4,Ride haling,blusmart,224,0.17
5,Ride haling,namma yatri,156,0.12
6,Ride haling,quick ride,78,0.06
7,Ride haling,driveu driver app,7,0.01
8,Ride haling,quickride,1,0.00
9,Office,zoom,27591,21.14


#### Transform Categories level-2

In [138]:
df_customer_mapped.head(1)

,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count
0,573f28f59b0ffc283676f94b,"[zomato, bookmyshow, kite by zerodha, uber, ju...",20,"[Tools, Educational, Delivery_Food, OTT, Enter...",10,"[Educational, Ride haling, Finance_Investment,...",4


In [139]:
unique_categories = set(category for sublist in df_customer_mapped['categories_l2'] for category in sublist)
# print(unique_categories)

for category in unique_categories:
    df_customer_mapped[category] = df_customer_mapped['categories_l2']\
                                            .apply(lambda x: 1 if category in x else 0)
##When havnig nan last column
# df_customer_mapped.drop(df_customer_mapped.columns[-1], axis=1, inplace=True)
df_customer_mapped.head(1)

,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest
0,573f28f59b0ffc283676f94b,"[zomato, bookmyshow, kite by zerodha, uber, ju...",20,"[Tools, Educational, Delivery_Food, OTT, Enter...",10,"[Educational, Ride haling, Finance_Investment,...",4,0,1,0,0,1,1,1


In [140]:
df_customer_mapped.to_csv(local_datasets + 'city/indore/processed/customer_app_categories_mapped_v1.csv',
                          index=False
                         )

### Customer-wise data dump

In [141]:
customer_mapped_final = pd.read_csv(local_datasets 
                                    + 'city/indore/processed/customer_app_categories_mapped_v1.csv')
print(customer_mapped_final.shape)
customer_mapped_final.head(5)

(130491, 14)


,customer_id,app_list,app_count,categories_l1,categories_l1_count,categories_l2,categories_l2_count,Driver_App,Educational,Gaming,Office,Finance_Investment,Ride haling,Rest
0,573f28f59b0ffc283676f94b,"['zomato', 'bookmyshow', 'kite by zerodha', 'u...",20,"['Tools', 'Educational', 'Delivery_Food', 'OTT...",10,"['Educational', 'Ride haling', 'Finance_Invest...",4,0,1,0,0,1,1,1
1,573f28fe9b0ffc28367719f1,"['zoom', 'linkedin', 'microsoft teams', 'netfl...",25,"['Tools', 'Social', 'News', 'OTT', 'Delivery_G...",10,"['Ride haling', 'Office', 'Rest']",3,0,0,0,1,0,1,1
2,573f29089b0ffc2836773ef0,"['telegram', 'zoom', 'cred', 'zomato', 'messen...",30,"['Tools', 'Social', 'News', 'Educational', 'De...",12,"['Educational', 'Finance_Investment', 'Office'...",5,0,1,0,1,1,1,1
3,573f290e9b0ffc2836775ef5,"['telegram', 'zomato', 'linkedin', 'indigo', '...",23,"['Tools', 'Fantasy_Sports', 'Social', 'Deliver...",12,"['Ride haling', 'Office', 'Rest']",3,0,0,0,1,0,1,1
4,573f292a9b0ffc283677b413,"['instagram', 'facebook', 'kite by zerodha', '...",6,"['OTT', 'Social', 'Finance_Investment', 'Deliv...",4,"['Finance_Investment', 'Rest']",2,0,0,0,0,1,0,1
